
# 02 — ResNet-50 Training for Lung Opacity Classification

## Project Overview

**Project:** When Stable Explanations Disagree: A Stability–Localization Disagreement Audit for Chest X-ray Classification

**Research question:** Can an explanation remain highly stable under image corruption while losing alignment with radiologist-annotated opacity regions?

**Intended audience:** Machine learning researchers, XAI practitioners, and developers evaluating explanation reliability in medical imaging AI.

### Purpose of This Notebook

This notebook fine-tunes a ResNet-50 classifier on the RSNA Pneumonia Detection Challenge dataset and establishes the fixed predictive model used in subsequent XAI experiments.

The primary contribution of the project is not a new classification architecture. Instead, the trained model serves as a controlled foundation for evaluating whether explanation stability and localization alignment can diverge under image corruption.

### Objectives

1. Verify the prepared training, validation, and test datasets.
2. Load and preprocess DICOM chest X-ray images.
3. Initialize ResNet-50 using pretrained chest X-ray weights.
4. Fine-tune the classifier for binary lung-opacity prediction.
5. Select the best model using validation AUROC.
6. Evaluate the model on held-out test images.
7. Save model checkpoints and predictions for subsequent explanation analyses.

### Overall Research Workflow

**Notebook 01 — Data Preparation:** Prepare image labels, DICOM files, radiologist annotations, patient identifiers, and dataset splits.

**Notebook 02 — Model Training:** Fine-tune and evaluate the ResNet-50 classifier.

**Notebooks 03–05 — Explanation Generation:** Generate Score-CAM, Integrated Gradients, and Occlusion Sensitivity explanations.

**Notebook 06 — Explanation Comparison:** Evaluate explanation stability and localization alignment across Gaussian blur and Gaussian noise conditions.

### Scope

The downstream XAI evaluation uses 300 held-out images (150 positive and 150 negative).

The primary stability–localization disagreement audit uses 150 annotated positive images across three explanation methods and six corruption conditions, resulting in 2,700 repeated evaluations.

This is an exploratory study of explanation behavior, not a clinical validation of the classifier.

## 1. Environment Setup

The experiment is implemented in Google Colab using PyTorch.

Google Drive stores the source data, model checkpoints, training history, and evaluation outputs. Local Colab storage is used to improve DICOM loading efficiency.

GPU acceleration is used when available.

The code and saved outputs document the experiment, although exact reproduction also depends on software versions, hardware, source files, and random states.


In [ ]:
# mount the notebook to google drive
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q pydicom

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pydicom

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

In [ ]:
PROJECT_DIR = Path(
    "/content/drive/MyDrive/XAI_project_Expo_1_folder"
)

DATA_DIR = PROJECT_DIR / "Data"
MODEL_DIR = PROJECT_DIR / "Models"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_PATH = DATA_DIR / "model_dataset_split.csv"

print("Project directory:", PROJECT_DIR)
print("Dataset:", SPLIT_PATH)
print("Dataset exists:", SPLIT_PATH.exists())
print("Model directory:", MODEL_DIR)

Project directory: /content/drive/MyDrive/XAI_project_Expo_1_folder
Dataset: /content/drive/MyDrive/XAI_project_Expo_1_folder/Data/model_dataset_split.csv
Dataset exists: True
Model directory: /content/drive/MyDrive/XAI_project_Expo_1_folder/Models


In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not available.")

Device: cuda
GPU: Tesla T4


In [ ]:
from pathlib import Path

LOCAL_EXTRACT_DIR = Path("/content/rsna_local/extracted")

print("Local DICOM files:",
      len(list(LOCAL_EXTRACT_DIR.rglob("*.dcm"))))

Local DICOM files: 30000


## 2. Dataset Inspection and Validation

We load the previously prepared dataset split file, which contains image paths, binary target labels, and training, validation, and test assignments.

Before model training, we inspect the dataset to verify:

- The number of observations assigned to each split.
- The distribution of positive and negative target labels.
- Whether the required image paths are available.

The target variable represents lung opacity classification:

- **0:** Negative for the target lung opacity label.
- **1:** Positive for the target lung opacity label.

The predefined data splits are maintained throughout training and evaluation.

The training set is used to optimize model parameters, the validation set is used to assess model performance during training, and the test set is reserved for final evaluation.


### Data Provenance and Reproducibility

**Dataset:** RSNA Pneumonia Detection Challenge (2018)

**Prepared input:** `Data/model_dataset_split.csv`

**Pretrained model repository:** `mamounyosef/chest-xray-bench`

The modeling dataset was prepared in Notebook 01 and contains 29,684 image records.

The original split assignments are reused:

| Dataset | Images | Purpose |
|---|---:|---|
| Training | 20,778 | Model optimization |
| Validation | 4,453 | Model selection |
| Test | 4,453 | Held-out evaluation |

The target is defined as:

- `1`: Lung Opacity
- `0`: Normal or No Lung Opacity / Not Normal

### Reproducibility Decisions

- Reuse the saved dataset splits rather than creating new assignments.
- Calculate class weights using the training subset only.
- Select the model using validation AUROC.
- Reserve the test subset for held-out evaluation.
- Save model checkpoints, training history, and predictions.
- Use the same selected model checkpoint for all three XAI methods.

### Reproduction Requirements

Researchers need access to the RSNA DICOM images, prepared dataset CSV, pretrained model weights, and compatible Python packages.

The current notebook uses Google Drive-specific paths and does not pin every dependency or pretrained repository revision.

Consequently, the saved model checkpoint is important for reproducing the downstream XAI analyses without requiring an identical retraining run.

The training, validation, and test assignments were inherited from Notebook 01, where patient-ID overlap checks confirmed that the saved subsets were patient-disjoint according to the available DICOM identifiers.

In [ ]:
model_df = pd.read_csv(SPLIT_PATH)

print("Rows:", len(model_df))
print("\nSplit counts:")
print(model_df["split"].value_counts())

print("\nTarget counts:")
print(model_df["target"].value_counts())

print("\nMissing image paths:")
print(model_df["image_path"].isna().sum())

Rows: 29684

Split counts:
split
train         20778
validation     4453
test           4453
Name: count, dtype: int64

Target counts:
target
0    22578
1     7106
Name: count, dtype: int64

Missing image paths:
0


## 3. DICOM Image Loading and Initial Preprocessing

The RSNA dataset contains chest X-ray images in DICOM format, which is commonly used for storing medical imaging data.

We define a custom image-loading function to convert DICOM images into numerical arrays suitable for machine learning.

### Preprocessing Steps

1. Read the DICOM file using `pydicom`.
2. Extract the pixel intensity array.
3. Correct inverted grayscale images when the DICOM photometric interpretation is `MONOCHROME1`.
4. Normalize pixel intensities.
5. Prepare the resulting image for the ResNet-50 preprocessing pipeline.

We initially define a standard ResNet-50 transformation and inspect the transformed image dimensions.

Later in the notebook, we replace this initial transformation with a medical-image-specific preprocessing pipeline that is used for model fine-tuning.

This distinction is important because the final model should use consistent preprocessing during training, evaluation, and subsequent explanation generation.

In [ ]:
def load_dicom(path):
    dcm = pydicom.dcmread(path)

    image = dcm.pixel_array.astype(np.float32)

    # Correct inverted grayscale images
    if getattr(dcm, "PhotometricInterpretation", "") == "MONOCHROME1":
        image = image.max() - image

    # Normalize pixel values to [0, 1]
    image_min = image.min()
    image_max = image.max()

    if image_max > image_min:
        image = (image - image_min) / (image_max - image_min)
    else:
        image = np.zeros_like(image)

    return image

In [ ]:
from torchvision import transforms

resnet_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
example_path = model_df.iloc[0]["image_path"]

image = load_dicom(example_path)
transformed_image = resnet_transform(image)

print("Original image shape:", image.shape)
print("ResNet input shape:", transformed_image.shape)
print("Data type:", transformed_image.dtype)
print("Contains NaN:", torch.isnan(transformed_image).any().item())

Original image shape: (1024, 1024)
ResNet input shape: torch.Size([3, 224, 224])
Data type: torch.float32
Contains NaN: False


## 4. Creating the PyTorch Dataset and DataLoaders

We define a custom PyTorch `Dataset` class to load chest X-ray images and their corresponding binary labels.

The dataset class retrieves an image using its stored file path, applies the specified preprocessing transformation, and returns the image tensor together with its target label.

Separate datasets and DataLoaders are constructed for training, validation, and testing.

The training DataLoader shuffles observations to randomize the order of training examples, while validation and test DataLoaders preserve a fixed ordering.

We initially use a batch size of 32 and inspect a sample batch to verify that the image tensors and labels have the expected dimensions and data types.

These checks help identify potential data-loading issues before beginning computationally expensive model training.

In [ ]:
from torch.utils.data import Dataset, DataLoader

class ChestXrayDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        image = load_dicom(row["image_path"])

        if self.transform is not None:
            image = self.transform(image)

        target = torch.tensor(
            row["target"],
            dtype=torch.float32
        )

        return image, target

In [ ]:
train_df = model_df[model_df["split"] == "train"].copy()
val_df = model_df[model_df["split"] == "validation"].copy()
test_df = model_df[model_df["split"] == "test"].copy()

train_dataset = ChestXrayDataset(
    train_df,
    transform=resnet_transform
)

val_dataset = ChestXrayDataset(
    val_df,
    transform=resnet_transform
)

test_dataset = ChestXrayDataset(
    test_df,
    transform=resnet_transform
)

print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Testing images:", len(test_dataset))

Training images: 20778
Validation images: 4453
Testing images: 4453


In [ ]:
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

In [ ]:
import time

start_time = time.perf_counter()

images, targets = next(iter(train_loader))

elapsed = time.perf_counter() - start_time

print("Image batch shape:", images.shape)
print("Target batch shape:", targets.shape)
print("Image dtype:", images.dtype)
print("Target dtype:", targets.dtype)
print(f"Time to load one batch: {elapsed:.2f} seconds")

Image batch shape: torch.Size([32, 3, 224, 224])
Target batch shape: torch.Size([32])
Image dtype: torch.float32
Target dtype: torch.float32
Time to load one batch: 49.63 seconds


## 5. Optimizing Data Loading with Local Storage

Reading large numbers of DICOM files directly from Google Drive can introduce substantial input/output overhead during model training.

To improve data-loading efficiency, we copy and extract the compressed dataset into the local Google Colab environment.

We then create a lookup table that maps DICOM image identifiers to their corresponding local file paths.

The dataset paths are updated in memory, and the DataLoaders are reconstructed to read from local storage.

Finally, we measure the time required to load a batch of images.

This optimization reduces dependence on Google Drive during training and can improve overall training throughput.

**Note:** Colab local storage is temporary, so the original dataset and important training artifacts remain stored in Google Drive.

In [ ]:
import shutil
from pathlib import Path

# Check available local storage
total, used, free = shutil.disk_usage("/content")

print(f"Colab total storage: {total / 1e9:.2f} GB")
print(f"Colab used storage: {used / 1e9:.2f} GB")
print(f"Colab free storage: {free / 1e9:.2f} GB")

# Check original compressed dataset size
ZIP_PATH = DATA_DIR / "pneumonia-challenge-dataset-adjudicated-kaggle_2018.zip"

if ZIP_PATH.exists():
    print(f"\nRSNA ZIP size: {ZIP_PATH.stat().st_size / 1e9:.2f} GB")
else:
    print("\nZIP not found at expected path.")

Colab total storage: 253.06 GB
Colab used storage: 45.60 GB
Colab free storage: 207.44 GB

RSNA ZIP size: 3.98 GB


In [ ]:
import shutil
import zipfile
import time

ZIP_PATH = DATA_DIR / "pneumonia-challenge-dataset-adjudicated-kaggle_2018.zip"

LOCAL_DIR = Path("/content/rsna_local")
LOCAL_ZIP = LOCAL_DIR / ZIP_PATH.name
LOCAL_EXTRACT_DIR = LOCAL_DIR / "extracted"

LOCAL_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# Copy the ZIP only if needed
if not LOCAL_ZIP.exists() or LOCAL_ZIP.stat().st_size != ZIP_PATH.stat().st_size:
    print("Copying ZIP from Google Drive to local Colab storage...", flush=True)
    start = time.perf_counter()
    shutil.copy2(ZIP_PATH, LOCAL_ZIP)
    print(f"Copy completed in {time.perf_counter() - start:.1f} seconds", flush=True)
else:
    print("Local ZIP already exists. Skipping copy.")

# Extract only if the expected DICOM count is not already present
existing_dicoms = list(LOCAL_EXTRACT_DIR.rglob("*.dcm"))

if len(existing_dicoms) != 30000:
    print(f"Found {len(existing_dicoms)} local DICOMs. Extracting archive...", flush=True)
    start = time.perf_counter()

    with zipfile.ZipFile(LOCAL_ZIP, "r") as zf:
        zf.extractall(LOCAL_EXTRACT_DIR)

    print(f"Extraction completed in {time.perf_counter() - start:.1f} seconds", flush=True)
else:
    print("All 30,000 local DICOMs already exist. Skipping extraction.")

print("Local DICOM count:", len(list(LOCAL_EXTRACT_DIR.rglob("*.dcm"))))

Copying ZIP from Google Drive to local Colab storage...
Copy completed in 116.8 seconds
Found 0 local DICOMs. Extracting archive...
Extraction completed in 22.1 seconds
Local DICOM count: 30000


In [ ]:
# Find the locally extracted DICOM images
local_dicom_paths = list(LOCAL_EXTRACT_DIR.rglob("*.dcm"))

# Map each SOP Instance UID to its local file path
local_path_lookup = {
    p.stem: str(p)
    for p in local_dicom_paths
}

# Update paths in memory, not in the saved CSV
model_df["image_path"] = model_df["sop_uid"].map(local_path_lookup)

print("Total labeled images:", len(model_df))
print("Images with valid local paths:", model_df["image_path"].notna().sum())
print("Missing local paths:", model_df["image_path"].isna().sum())

assert model_df["image_path"].notna().all(), "Some local paths are missing."

Total labeled images: 29684
Images with valid local paths: 29684
Missing local paths: 0


In [ ]:
train_df = model_df[model_df["split"] == "train"].copy()
val_df = model_df[model_df["split"] == "validation"].copy()
test_df = model_df[model_df["split"] == "test"].copy()

train_dataset = ChestXrayDataset(train_df, transform=resnet_transform)
val_dataset = ChestXrayDataset(val_df, transform=resnet_transform)
test_dataset = ChestXrayDataset(test_df, transform=resnet_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print("Datasets and DataLoaders rebuilt successfully.")

Datasets and DataLoaders rebuilt successfully.


In [ ]:
import time

start = time.perf_counter()

images, targets = next(iter(train_loader))

elapsed = time.perf_counter() - start

print("Image batch shape:", images.shape)
print("Target batch shape:", targets.shape)
print(f"Local batch-loading time: {elapsed:.2f} seconds")

Image batch shape: torch.Size([32, 3, 224, 224])
Target batch shape: torch.Size([32])
Local batch-loading time: 0.39 seconds


## 6. Loading a Pretrained Medical Imaging Model

Instead of initializing ResNet-50 with random weights, we use a pretrained model from the Hugging Face repository `mamounyosef/chest-xray-bench`.

The selected checkpoint, `resnet50_without_clahe`, contains weights from a ResNet-50 model pretrained on chest X-ray data.

### Model Adaptation

The pretrained architecture originally produces five classification outputs. To adapt it to our binary lung opacity classification task, we:

1. Download the model configuration and pretrained weights.
2. Reconstruct the original five-output ResNet-50 architecture.
3. Load the pretrained parameters.
4. Replace the final fully connected layer with a single-output classification layer.
5. Transfer the modified model to the available computing device.

The resulting model produces one logit per image, which can be converted into a probability using the sigmoid function.

This transfer learning approach allows us to leverage previously learned medical imaging features while adapting the model to the RSNA classification task.

In [ ]:
!pip -q install huggingface_hub safetensors timm

In [ ]:
from huggingface_hub import hf_hub_download
import json

REPO_ID = "mamounyosef/chest-xray-bench"
MODEL_NAME = "resnet50_without_clahe"

config_path = hf_hub_download(
    repo_id=REPO_ID,
    filename=f"{MODEL_NAME}/config.json"
)

with open(config_path, "r") as f:
    medical_config = json.load(f)

print(json.dumps(medical_config, indent=2))

config.json:   0%|          | 0.00/1.70k [00:00<?, ?B/s]

{
  "run": "resnet50_without_clahe",
  "backbone": "resnet50",
  "arch": null,
  "tasks": [
    "Atelectasis",
    "Cardiomegaly",
    "Consolidation",
    "Edema",
    "Pleural Effusion"
  ],
  "image": {
    "width": 384,
    "height": 320,
    "norm_mean": [
      0.485,
      0.456,
      0.406
    ],
    "norm_std": [
      0.229,
      0.224,
      0.225
    ],
    "interpolation": "INTER_AREA"
  },
  "labels": {
    "u_policy": "ones"
  },
  "label_smoothing": null,
  "source_checkpoint": "results\\checkpoints\\best.pt",
  "n_parameters": 23571450,
  "head": {
    "n_logits": 5,
    "layout": "one sigmoid logit per task",
    "task_slices": {
      "Atelectasis": [
        0,
        1
      ],
      "Cardiomegaly": [
        1,
        2
      ],
      "Consolidation": [
        2,
        3
      ],
      "Edema": [
        3,
        4
      ],
      "Pleural Effusion": [
        4,
        5
      ]
    }
  },
  "results": {
    "valid200": {
      "n_images": 202,
      "me

In [ ]:
from huggingface_hub import list_repo_files

REPO_ID = "mamounyosef/chest-xray-bench"
MODEL_NAME = "resnet50_without_clahe"

repo_files = list_repo_files(REPO_ID)

model_files = [
    filename
    for filename in repo_files
    if MODEL_NAME in filename
]

for filename in model_files:
    print(filename)

resnet50_without_clahe/config.json
resnet50_without_clahe/model.safetensors
resnet50_without_clahe/thresholds.json


In [ ]:
from huggingface_hub import hf_hub_download
from safetensors import safe_open

weights_path = hf_hub_download(
    repo_id=REPO_ID,
    filename=f"{MODEL_NAME}/model.safetensors"
)

print("Weights downloaded successfully!")
print("Weights location:", weights_path)

resnet50_without_clahe/model.safetensors: reconstructing file:   0%|          |  0.00B / 94.3MB            

resnet50_without_clahe/model.safetensors: downloading bytes:           |  0.00B            

Weights downloaded successfully!
Weights location: /root/.cache/huggingface/hub/models--mamounyosef--chest-xray-bench/snapshots/1735afb88371d98f87667c5f9966ed47bf4ef678/resnet50_without_clahe/model.safetensors


In [ ]:
with safe_open(weights_path, framework="pt", device="cpu") as f:
    keys = list(f.keys())

    print("Number of weight tensors:", len(keys))

    print("\nFirst 15 tensor names:")
    for key in keys[:15]:
        print(key)

    print("\nClassification-related tensors:")
    for key in keys:
        if any(term in key.lower() for term in ["fc.", "classifier", "head."]):
            print(key, f.get_slice(key).get_shape())

Number of weight tensors: 320

First 15 tensor names:
bn1.bias
bn1.num_batches_tracked
bn1.running_mean
bn1.running_var
bn1.weight
conv1.weight
fc.bias
fc.weight
layer1.0.bn1.bias
layer1.0.bn1.num_batches_tracked
layer1.0.bn1.running_mean
layer1.0.bn1.running_var
layer1.0.bn1.weight
layer1.0.bn2.bias
layer1.0.bn2.num_batches_tracked

Classification-related tensors:
fc.bias [5]
fc.weight [5, 2048]


In [ ]:
import torch
import torch.nn as nn
from torchvision.models import resnet50
from safetensors.torch import load_file

# 1. Load the medical checkpoint
medical_weights = load_file(weights_path, device="cpu")

# 2. Reconstruct the original 5-output ResNet-50
model = resnet50(weights=None, num_classes=5)

# 3. Load all pretrained weights
model.load_state_dict(medical_weights, strict=True)

print("Medical pretrained weights loaded successfully!")

# 4. Replace the 5-output head with a binary classifier
model.fc = nn.Linear(model.fc.in_features, 1)

# 5. Move to GPU
model = model.to(device)
model.eval()

print("Final classification layer:", model.fc)
print("Device:", device)

Medical pretrained weights loaded successfully!
Final classification layer: Linear(in_features=2048, out_features=1, bias=True)
Device: cuda


In [ ]:
MODEL_DIR = Path(
    "/content/drive/MyDrive/XAI_project_Expo_1_folder/Models"
)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

initial_model_path = MODEL_DIR / "resnet50_chexpert_initial.pth"

torch.save({
    "model_state_dict": model.state_dict(),
    "pretraining": "CheXpert",
    "architecture": "ResNet50",
    "num_outputs": 1,
    "trained_on_rsna": False
}, initial_model_path)

print("Model saved to:", initial_model_path)
print("File exists:", initial_model_path.exists())

Model saved to: /content/drive/MyDrive/XAI_project_Expo_1_folder/Models/resnet50_chexpert_initial.pth
File exists: True



## 7. Medical Image Preprocessing

The pretrained model requires a consistent input representation for fine-tuning and evaluation.

### Preprocessing Pipeline

The `MedicalImageTransform` class performs the following operations:

1. Resizes each DICOM image to 384 × 320 pixels (width × height).
2. Replicates the grayscale image across three channels.
3. Converts the image to a floating-point PyTorch tensor.
4. Applies ImageNet channel-wise normalization.

The final tensor shape is:

`[3, 320, 384]`

The DICOM loading function defined earlier also corrects `MONOCHROME1` images and performs per-image intensity normalization.

### Connection to the XAI Experiments

The explanation methods compare attribution maps generated from original and corrupted chest X-rays.

Using consistent preprocessing is important because changes in resizing, intensity scaling, or normalization could influence model predictions and explanations independently of the intended corruption.

For localization analysis, attribution maps and radiologist bounding boxes must also be represented in a consistent coordinate system.

### Execution Note

The notebook initially constructs DataLoaders using a standard 224 × 224 ResNet transformation.

This section introduces the medical-image preprocessing pipeline and rebuilds the validation DataLoader.

Following the recorded Colab interruption, Section 11 reconstructs all three DataLoaders using `medical_transform` and a batch size of 16.

The final evaluation is recorded after that reconstruction.

### Reproducibility Limitation

The current notebook contains both the initial exploratory preprocessing and the later training configuration.

A clean reproduction should use the final medical-image transformation consistently for training, validation, and testing from the beginning.


In [ ]:
import cv2
import numpy as np
import torch
from torchvision import transforms

class MedicalImageTransform:
    def __init__(self):
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )

    def __call__(self, image):
        # Resize to width=384, height=320
        image = cv2.resize(
            np.asarray(image, dtype=np.float32),
            (384, 320),
            interpolation=cv2.INTER_AREA
        )

        # Convert grayscale image to 3 channels
        image = np.stack([image, image, image], axis=0)

        # Convert to PyTorch tensor
        image = torch.from_numpy(image).float()

        # Apply pretrained model normalization
        return self.normalize(image)

medical_transform = MedicalImageTransform()

In [ ]:
sample_path = model_df.iloc[0]["image_path"]

sample_image = load_dicom(sample_path)
sample_tensor = medical_transform(sample_image)

print("Original shape:", sample_image.shape)
print("Transformed shape:", sample_tensor.shape)
print("Data type:", sample_tensor.dtype)
print("Contains NaN:", torch.isnan(sample_tensor).any().item())

Original shape: (1024, 1024)
Transformed shape: torch.Size([3, 320, 384])
Data type: torch.float32
Contains NaN: False


In [ ]:
# Correct the validation split name
val_df = model_df[
    model_df["split"] == "validation"
].reset_index(drop=True)

# Recreate the validation dataset
val_dataset = ChestXrayDataset(
    val_df,
    transform=medical_transform
)

# Recreate the validation DataLoader
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

# Verify all three datasets
print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Test images:", len(test_dataset))

assert len(train_dataset) == 20778
assert len(val_dataset) == 4453
assert len(test_dataset) == 4453

print("All dataset sizes verified!")

Training images: 20778
Validation images: 4453
Test images: 4453
All dataset sizes verified!


In [ ]:
import time
import torch

start = time.time()

# Load one batch
images, labels = next(iter(train_loader))

loading_time = time.time() - start

print("Batch loading time:", round(loading_time, 2), "seconds")
print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)

# Test the model without training
model.eval()

with torch.no_grad():
    outputs = model(images.to(device))

print("Model output shape:", outputs.shape)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Batch loading time: 0.16 seconds
Image batch shape: torch.Size([16, 3, 320, 384])
Label batch shape: torch.Size([16])
Model output shape: torch.Size([16, 1])
GPU: Tesla T4


## 8. Training Configuration

We configure the loss function and optimizer for binary lung opacity classification.

### Handling Class Imbalance

Because the number of positive and negative observations may differ, we calculate a positive-class weight using the training dataset:

`pos_weight = n_negative / n_positive`

We incorporate this weight into `BCEWithLogitsLoss`, which combines sigmoid activation and binary cross-entropy loss in a numerically stable implementation.

This weighting increases the relative contribution of positive examples when the positive class is less frequent.


### Training Reproducibility and Experimental Controls

The classifier is fine-tuned using the following configuration:

| Component | Configuration |
|---|---|
| Architecture | ResNet-50 |
| Initialization | Pretrained chest X-ray weights |
| Output | Single binary classification logit |
| Loss | Weighted `BCEWithLogitsLoss` |
| Optimizer | AdamW |
| Learning rate | 0.0001 |
| Weight decay | 0.0001 |
| Fine-tuning | All model parameters trainable |
| Model selection | Highest validation AUROC |
| Final recovery batch size | 16 |

The positive-class weight is calculated using training observations only.

The recorded training subset contains 4,974 positive images and 15,804 negative images, giving a positive-class weight of approximately 3.177.

### Reproducibility Considerations

The dataset splits are fixed and saved, but the notebook does not establish complete deterministic training.

Sources of variation may include:

- Random initialization of the replacement classification layer.
- Training DataLoader shuffling.
- GPU computation and nondeterministic operations.
- Software and library versions.
- Runtime interruptions and checkpoint recovery.

The notebook preserves the trained checkpoint and evaluation outputs so that downstream explanation experiments can reuse the selected classifier.

These saved artifacts support reproducibility of the XAI analysis, even if an independently retrained model does not have identical parameters.


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

# Calculate class imbalance using TRAINING data only
n_positive = int((train_df["target"] == 1).sum())
n_negative = int((train_df["target"] == 0).sum())

pos_weight = torch.tensor(
    [n_negative / n_positive],
    dtype=torch.float32,
    device=device
)

# Binary classification loss
criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

# Fine-tune all ResNet-50 layers
optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

print("Positive training cases:", n_positive)
print("Negative training cases:", n_negative)
print("Positive class weight:", round(pos_weight.item(), 3))
print("Loss function:", criterion)
print("Optimizer:", type(optimizer).__name__)

NameError: name 'train_df' is not defined


## 9. Model Checkpointing and Training History

To preserve training progress and support reproducibility, model checkpoints and training history are saved to Google Drive.

Three primary artifacts are maintained:

- **`resnet50_best.pth`:** Stores the checkpoint corresponding to the highest validation AUROC achieved during training.
- **`resnet50_checkpoint.pth`:** Stores the checkpoint from the most recently completed training epoch.
- **`training_history.csv`:** Records training loss, validation loss, validation AUROC, and training duration for each completed epoch.

Both checkpoint files contain the model parameters, optimizer state, epoch information, and training metadata. The best-model checkpoint is selected using validation AUROC.

### Why Use Validation AUROC?

AUROC evaluates how well the classifier distinguishes positive and negative cases across different decision thresholds.

Selecting the model based on validation AUROC provides a consistent model-selection criterion without using the held-out test dataset to choose the checkpoint.

### Relevance to the XAI Experiments

The best-performing validation checkpoint is subsequently loaded for test evaluation and all three XAI methods.

Using the same trained model across Score-CAM, Integrated Gradients, and Occlusion Sensitivity ensures that differences in explanations are not caused by changes in model parameters.


In [ ]:
from pathlib import Path
import torch
import pandas as pd

MODEL_DIR = Path(
    "/content/drive/MyDrive/XAI_project_Expo_1_folder/Models"
)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = MODEL_DIR / "resnet50_best.pth"
CHECKPOINT_PATH = MODEL_DIR / "resnet50_checkpoint.pth"
HISTORY_PATH = MODEL_DIR / "training_history.csv"

# Initial training state
start_epoch = 0
best_val_auc = float("-inf")
training_history = []

def save_training_checkpoint(epoch, val_auc, is_best=False):
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_val_auc": best_val_auc,
        "training_history": training_history,
        "pretraining": "CheXpert",
        "architecture": "ResNet50",
        "image_size": (320, 384)
    }

    # Save latest completed epoch
    torch.save(checkpoint, CHECKPOINT_PATH)

    # Save best validation model separately
    if is_best:
        torch.save(checkpoint, BEST_MODEL_PATH)

    pd.DataFrame(training_history).to_csv(
        HISTORY_PATH,
        index=False
    )

print("Checkpoint directory:", MODEL_DIR)
print("Latest checkpoint:", CHECKPOINT_PATH.name)
print("Best model:", BEST_MODEL_PATH.name)
print("Training history:", HISTORY_PATH.name)
print("Checkpoint system configured!")

Checkpoint directory: /content/drive/MyDrive/XAI_project_Expo_1_folder/Models
Latest checkpoint: resnet50_checkpoint.pth
Best model: resnet50_best.pth
Training history: training_history.csv
Checkpoint system configured!


In [ ]:
print("Model directory exists:", MODEL_DIR.exists())
print("Initial model exists:",
      (MODEL_DIR / "resnet50_chexpert_initial.pth").exists())

print("Training begins at epoch:", start_epoch)
print("Best validation AUROC:", best_val_auc)

Model directory exists: True
Initial model exists: True
Training begins at epoch: 0
Best validation AUROC: -inf


## 10. Training and Validation Functions

We implement separate functions for training and validating the ResNet-50 model.

### Training

During each training epoch, the model:

1. Receives batches of preprocessed chest X-ray images.
2. Generates classification logits.
3. Calculates the weighted binary cross-entropy loss.
4. Computes gradients through backpropagation.
5. Updates model parameters using AdamW.

### Validation

After training, the model is evaluated on the validation dataset without updating its parameters.

We calculate validation loss and Area Under the Receiver Operating Characteristic Curve (AUROC).

AUROC measures the model's ability to rank positive examples above negative examples across different classification thresholds.

The validation results are used to track performance and identify the best model checkpoint.

In [ ]:
from sklearn.metrics import roc_auc_score

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    total_images = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device).float().view(-1, 1)

        optimizer.zero_grad(set_to_none=True)

        logits = model(images)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        batch_size = images.size(0)
        total_loss += loss.item() * batch_size
        total_images += batch_size

    return total_loss / total_images


@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    total_images = 0

    all_labels = []
    all_probabilities = []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device).float().view(-1, 1)

        logits = model(images)
        loss = criterion(logits, labels)

        probabilities = torch.sigmoid(logits)

        batch_size = images.size(0)
        total_loss += loss.item() * batch_size
        total_images += batch_size

        all_labels.extend(labels.cpu().numpy().ravel())
        all_probabilities.extend(
            probabilities.cpu().numpy().ravel()
        )

    val_loss = total_loss / total_images

    val_auc = roc_auc_score(
        all_labels,
        all_probabilities
    )

    return val_loss, val_auc


print("Training and validation functions defined successfully!")

Training and validation functions defined successfully!


In [ ]:
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("GPU:", device)
print("Checkpoint directory:", MODEL_DIR)

Training batches: 1299
Validation batches: 279
GPU: cuda
Checkpoint directory: /content/drive/MyDrive/XAI_project_Expo_1_folder/Models


In [ ]:
import time
import torch

model.train()

# Save the current random-number state so the benchmark
# doesn't affect the subsequent training sequence.
cpu_rng_state = torch.get_rng_state()
cuda_rng_state = torch.cuda.get_rng_state_all()

batch_times = []

try:
    for batch_idx, (images, labels) in enumerate(train_loader):
        if batch_idx >= 5:
            break

        images = images.to(device)
        labels = labels.to(device).float().view(-1, 1)

        optimizer.zero_grad(set_to_none=True)

        torch.cuda.synchronize()
        start = time.perf_counter()

        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()

        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start

        batch_times.append(elapsed)

        print(f"Batch {batch_idx + 1}: {elapsed:.2f} seconds")

finally:
    optimizer.zero_grad(set_to_none=True)
    torch.set_rng_state(cpu_rng_state)
    torch.cuda.set_rng_state_all(cuda_rng_state)
    model.eval()

# Exclude the first batch because GPU warm-up can distort timing
average_time = sum(batch_times[1:]) / len(batch_times[1:])

estimated_minutes = (
    average_time * len(train_loader) / 60
)

print(f"\nAverage compute time per batch: {average_time:.2f} seconds")
print(f"Estimated training compute time per epoch: {estimated_minutes:.1f} minutes")

Batch 1: 0.67 seconds
Batch 2: 0.36 seconds
Batch 3: 0.36 seconds
Batch 4: 0.36 seconds
Batch 5: 0.36 seconds

Average compute time per batch: 0.36 seconds
Estimated training compute time per epoch: 7.8 minutes


In [ ]:
import torch

initial_model_path = (
    MODEL_DIR / "resnet50_chexpert_initial.pth"
)

checkpoint = torch.load(
    initial_model_path,
    map_location=device,
    weights_only=True
)

model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

# Clear gradients left by the benchmark
optimizer.zero_grad(set_to_none=True)

model.eval()

print("Initial model restored successfully!")
print("Training has not started.")

Initial model restored successfully!
Training has not started.


In [ ]:
def resume_training(checkpoint_path=CHECKPOINT_PATH):
    global start_epoch, best_val_auc, training_history

    if not checkpoint_path.exists():
        print("No training checkpoint found. Starting from epoch 1.")
        return

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=True
    )

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    start_epoch = checkpoint["epoch"]
    best_val_auc = checkpoint["best_val_auc"]
    training_history = checkpoint["training_history"]

    print(f"Restored completed epoch: {start_epoch}")
    print(f"Best validation AUROC: {best_val_auc:.4f}")
    print(f"Next epoch: {start_epoch + 1}")


print("Resume function configured successfully!")

Resume function configured successfully!


In [ ]:
def run_training(num_epochs=1):
    global start_epoch, best_val_auc, training_history

    for epoch in range(start_epoch, start_epoch + num_epochs):
        print(f"\nEpoch {epoch + 1}")
        print("-" * 40)

        start_time = time.time()

        # Train using training images only
        train_loss = train_one_epoch(
            model, train_loader, criterion, optimizer, device
        )

        # Evaluate using validation images only
        val_loss, val_auc = validate_one_epoch(
            model, val_loader, criterion, device
        )

        elapsed_minutes = (time.time() - start_time) / 60

        print(f"Training loss: {train_loss:.4f}")
        print(f"Validation loss: {val_loss:.4f}")
        print(f"Validation AUROC: {val_auc:.4f}")
        print(f"Epoch duration: {elapsed_minutes:.1f} minutes")

        # Determine whether this is the best model so far
        is_best = np.isfinite(val_auc) and val_auc > best_val_auc

        if is_best:
            best_val_auc = val_auc

        training_history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_auc": val_auc,
            "duration_minutes": elapsed_minutes
        })

        # Save model, optimizer, and training history to Drive
        save_training_checkpoint(
            epoch=epoch + 1,
            val_auc=val_auc,
            is_best=is_best
        )

        start_epoch = epoch + 1

        print("Latest checkpoint saved.")

        if is_best:
            print("New best validation model saved!")

    print("\nTraining run completed.")


print("Training loop configured successfully!")

Training loop configured successfully!



## 11. Training Execution and Runtime Recovery

The ResNet-50 classifier is fine-tuned using the training subset, with performance monitored on the validation subset.

The model checkpoint achieving the highest validation AUROC is retained for final evaluation.

### Google Colab Runtime Interruption

During the experiment, the Google Colab runtime was disconnected.

The following cells reconstruct the required environment by:

1. Rebuilding the training, validation, and test DataLoaders.
2. Applying `medical_transform` to all three subsets.
3. Setting the batch size to 16.
4. Reconstructing the single-output ResNet-50 architecture.
5. Restoring the saved initial model parameters.
6. Checking the availability of training checkpoints.

These cells document recovery from an interruption rather than a separate modeling experiment.

### Recorded Training Outputs

The checkpoint inspection reports that the initial model file exists, while the latest training checkpoint is absent at that point.

A subsequent training cell displays:

- Epoch label: 2
- Training loss: 0.6427
- Validation loss: 0.6650
- Validation AUROC: 0.8791

The selected best checkpoint later reports:

- Best checkpoint epoch: 1
- Best validation AUROC: 0.8864

The best checkpoint is therefore selected from the available saved model artifacts rather than assuming that the last displayed epoch produced the best model.

### Reproducibility Limitation

The saved outputs do not fully establish the complete sequence of training and checkpoint restoration following the runtime interruption.

In particular, the absence of the latest checkpoint at the recorded inspection point means that the later training cell should not be described as a verified optimizer-state continuation.

For an independent reproduction, the training sequence, model initialization, DataLoaders, and checkpoint restoration should be executed in a clearly documented order.

### Final Model Selection

The subsequent evaluation explicitly loads `resnet50_best.pth`, ensuring that the selected saved checkpoint is used for the reported test results.


In [ ]:
## notebook got disconncted so had to run this:

from torch.utils.data import DataLoader

train_df = model_df[model_df["split"] == "train"].reset_index(drop=True)
val_df = model_df[model_df["split"] == "validation"].reset_index(drop=True)
test_df = model_df[model_df["split"] == "test"].reset_index(drop=True)

BATCH_SIZE = 16

train_dataset = ChestXrayDataset(train_df, transform=medical_transform)
val_dataset = ChestXrayDataset(val_df, transform=medical_transform)
test_dataset = ChestXrayDataset(test_df, transform=medical_transform)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE,
    shuffle=True, num_workers=0, pin_memory=True
)

val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=0, pin_memory=True
)

test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=0, pin_memory=True
)

print(len(train_dataset), len(val_dataset), len(test_dataset))

20778 4453 4453


In [ ]:
# restore the saved mdeical model

import torch
import torch.nn as nn
from torchvision.models import resnet50

model = resnet50(weights=None, num_classes=1).to(device)

initial_path = MODEL_DIR / "resnet50_chexpert_initial.pth"

saved = torch.load(
    initial_path,
    map_location=device,
    weights_only=True
)

model.load_state_dict(saved["model_state_dict"], strict=True)
model.eval()

print("Medical model restored successfully!")


Medical model restored successfully!


In [ ]:
print("Initial model:", initial_path.exists())
print("Training checkpoint:", CHECKPOINT_PATH.exists())
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Start epoch:", start_epoch)


Initial model: True
Training checkpoint: False
Training batches: 1299
Validation batches: 279
Start epoch: 0


In [ ]:
# training the model
run_training(num_epochs=1)


Epoch 2
----------------------------------------
Training loss: 0.6427
Validation loss: 0.6650
Validation AUROC: 0.8791
Epoch duration: 12.4 minutes
Latest checkpoint saved.

Training run completed.



## 12. Final Model Evaluation

The model checkpoint with the highest recorded validation AUROC is loaded for evaluation on the held-out test set.

### Evaluation Procedure

1. Load `resnet50_best.pth`.
2. Switch the model to evaluation mode.
3. Generate logits for the held-out test images.
4. Apply the sigmoid function to obtain positive-class probabilities.
5. Calculate AUROC, accuracy, and the confusion matrix.

Predicted classes are assigned using a probability threshold of 0.5.

### Recorded Results

| Metric | Value |
|---|---:|
| Best validation AUROC | 0.8864 |
| Test AUROC | 0.8793 |
| Test accuracy | 0.7777 |
| True negatives | 2,566 |
| False positives | 821 |
| False negatives | 169 |
| True positives | 897 |

### Interpretation

The model demonstrates an ability to distinguish the dataset-defined lung-opacity classes on held-out images.

AUROC evaluates discrimination across probability thresholds, while accuracy and the confusion matrix describe performance at the selected 0.5 threshold.

These metrics establish the classifier's predictive performance but do not determine whether its explanations are reliable.

A model can maintain its predicted class while the spatial distribution of an explanation changes under image corruption.

### Limitations

The evaluation uses one dataset and one model architecture.

The target represents dataset-defined lung opacity rather than comprehensive clinical pneumonia diagnosis.

The results do not establish clinical readiness, external generalizability, or explanation faithfulness.


In [ ]:
best_checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=device,
    weights_only=False
)

model.load_state_dict(best_checkpoint["model_state_dict"])
model.eval()

print("Best model loaded successfully!")
print("Best model epoch:", best_checkpoint["epoch"])
print("Best validation AUROC:", best_checkpoint["best_val_auc"])

Best model loaded successfully!
Best model epoch: 1
Best validation AUROC: 0.8864090211386545


In [ ]:
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
import numpy as np
import torch

model.eval()

all_labels = []
all_probabilities = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)

        logits = model(images)
        probabilities = torch.sigmoid(logits)

        all_labels.extend(labels.numpy().ravel())
        all_probabilities.extend(
            probabilities.cpu().numpy().ravel()
        )

y_true = np.array(all_labels)
y_prob = np.array(all_probabilities)

test_auc = roc_auc_score(y_true, y_prob)
y_pred = (y_prob >= 0.5).astype(int)

print(f"Test AUROC: {test_auc:.4f}")
print(f"Test accuracy (threshold=0.5): {accuracy_score(y_true, y_pred):.4f}")
print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred))

Test AUROC: 0.8793
Test accuracy (threshold=0.5): 0.7777
Confusion matrix:
[[2566  821]
 [ 169  897]]



## 13. Saving Test Predictions and Verifying Outputs

Following held-out evaluation, image-level predictions are saved to:

`Results/resnet50_test_predictions.csv`

The file contains the test image records, ground-truth targets, predicted probabilities, and predicted classes.

### Purpose

Saving predictions allows subsequent XAI notebooks to associate explanations with the same model outputs and image identifiers.

It also avoids unnecessarily repeating model training or test inference.

### Initial XAI Pilot

The initial explanation-generation workflow used 12 correctly classified test images:

- Six true positives.
- Six true negatives.

This pilot was used to establish and inspect the explanation-generation procedures.

It should not be confused with the final XAI evaluation.

### Final XAI Evaluation

The broader evaluation uses 300 held-out images:

- 150 positive images.
- 150 negative images.

The primary stability–localization disagreement audit focuses on 150 annotated positive images.

Each image is evaluated using three explanation methods and six corruption conditions:

`150 × 3 × 6 = 2,700 evaluations`

These represent repeated method–image–corruption observations, not 2,700 independent patients.

### Saved Artifacts

The notebook verifies the existence of:

- `Models/resnet50_best.pth`
- `Models/resnet50_checkpoint.pth`
- `Models/training_history.csv`
- `Results/resnet50_test_predictions.csv`

### Reproducibility

The selected model checkpoint and test predictions should be preserved alongside the image-selection records used in later notebooks.

This allows the explanation analyses to be traced back to a specific classifier and held-out evaluation set.

### Limitations

The 300-image XAI evaluation subset is smaller than the full 4,453-image test set.

The initial pilot also selects correctly classified images, so it should not be treated as a representative evaluation of model errors.

The final disagreement audit is restricted to annotated positive images because the localization metric requires opacity reference regions.


In [ ]:
test_results = test_df.copy()

test_results["predicted_probability"] = y_prob
test_results["predicted_class"] = y_pred

RESULTS_DIR = PROJECT_DIR / "Results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

test_results_path = RESULTS_DIR / "resnet50_test_predictions.csv"

test_results.to_csv(test_results_path, index=False)

print("Test predictions saved:", test_results_path.exists())
print("Number of predictions:", len(test_results))

Test predictions saved: True
Number of predictions: 4453


In [ ]:
# verify the saved files

files_to_check = {
    "Best trained model": MODEL_DIR / "resnet50_best.pth",
    "Latest training checkpoint": MODEL_DIR / "resnet50_checkpoint.pth",
    "Training history": MODEL_DIR / "training_history.csv",
    "Test predictions": RESULTS_DIR / "resnet50_test_predictions.csv",
}

for name, path in files_to_check.items():
    print(f"{name}: {'Saved' if path.exists() else 'MISSING'}")

Best trained model: Saved
Latest training checkpoint: Saved
Training history: Saved
Test predictions: Saved



## 14. Conclusion and Connection to the XAI Audit

This notebook establishes the fixed ResNet-50 classifier used in the project, *When Stable Explanations Disagree*.

The model was initialized using pretrained chest X-ray weights, adapted for binary lung-opacity classification, and evaluated on held-out RSNA images.

### Model Performance

The selected model achieved:

- Best validation AUROC: 0.8864
- Test AUROC: 0.8793
- Test accuracy: 0.7777

These results provide the predictive modeling foundation for the subsequent explanation experiments.

### Project Contribution

The primary research contribution is not the classifier itself.

Instead, the project investigates whether an explanation can remain stable under image corruption while losing spatial alignment with radiologist-annotated opacity regions.

The fixed classifier allows the three XAI methods to be evaluated without changing model parameters between experiments.

### Subsequent Evaluation

The next notebooks apply:

1. Score-CAM.
2. Integrated Gradients.
3. Occlusion Sensitivity.

The methods are evaluated under Gaussian blur and Gaussian noise, each at three severity levels.

Two complementary quantities are examined:

**Explanation stability:** Spearman correlation between original and corrupted absolute attribution rankings.

**Localization alignment:** Proportion of absolute attribution magnitude inside radiologist-annotated opacity regions.

The primary disagreement criterion flags an evaluation when:

- Explanation stability is at least 0.80.
- Localization alignment decreases by at least 0.05.

These thresholds are exploratory and have not been clinically validated.

### Reproducibility Summary

The notebook preserves the prepared dataset splits, pretrained model initialization, selected model checkpoint, training history, and held-out test predictions.

The Google Colab runtime interruption and incomplete environment versioning limit exact reproduction of the training trajectory.

However, preserving the selected checkpoint supports consistent reuse of the trained classifier across downstream XAI experiments.

### Limitations

- One model architecture and one chest X-ray dataset.
- Binary lung-opacity classification rather than comprehensive clinical diagnosis.
- No external clinical validation.
- No demonstration that attribution stability or bounding-box alignment establishes explanation faithfulness.

### Next Steps

Use the saved classifier to generate explanations and conduct the stability–localization disagreement audit.

The final analysis evaluates whether explanation stability alone is sufficient to assess preserved localization alignment under image corruption.
